# Aula 03 — Autograd: de escalares a VJPs

Compare a regra da cadeia manual com diferenciação automática reversa. CPU, float64, seed 20260903. Dados sintéticos inteiramente definidos aqui; não há treinamento, downloads ou credenciais.

Dependências mínimas: Python 3.10, NumPy 1.24, PyTorch 2.6. Referência executada: Python 3.12.14, NumPy 2.3.5, PyTorch 2.6.0+cpu. Execute todas as células em ordem. Para reproduzir o ambiente de referência em Python 3.12, use no terminal `python -m pip install numpy==2.3.5 torch==2.6.0` (a seleção de build/dispositivo depende da plataforma). Em Python 3.10, escolha versões compatíveis com esse interpretador e com os mínimos declarados. Reinicie o kernel após instalar.

[Aula em Markdown](../aulas/03-autograd-vjp.md). Este experimento verifica derivadas locais; não estima generalização e não requer splits artificiais.

## 1. Ambiente e assertivas
As tolerâncias são específicas destas fixtures pequenas. Cada comparação verifica também shape e finitude.

In [ ]:
import platform
import numpy as np
import torch
SEED = 20260903
DTYPE = torch.float64
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
checks = []
def close(name, actual, expected, atol=1e-12, rtol=1e-12):
    assert name not in checks
    expected = torch.as_tensor(expected, dtype=actual.dtype, device=actual.device)
    assert actual.shape == expected.shape, (name, actual.shape, expected.shape)
    assert torch.isfinite(actual).all(), name
    torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol)
    checks.append(name)
def ok(name, condition):
    assert name not in checks and bool(condition), name
    checks.append(name)
print(platform.python_version(), np.__version__, torch.__version__)


## 2. Escalar: derivar não atualiza o parâmetro
Para L=(wx+b)²/2 em w=2, x=3, b=1, temos L=24,5, dL/dw=21 e dL/db=7.

In [ ]:
w = torch.tensor(2., dtype=DTYPE, requires_grad=True)
b = torch.tensor(1., dtype=DTYPE, requires_grad=True)
z = 3 * w + b
loss = z.square() / 2
ok("folhas_e_operacao", w.is_leaf and b.is_leaf and z.grad_fn is not None)
ok("grad_antes", w.grad is None and b.grad is None)
loss.backward()
close("loss_escalar", loss.detach(), torch.tensor(24.5, dtype=DTYPE))
close("grad_w", w.grad, torch.tensor(21., dtype=DTYPE))
close("grad_b", b.grad, torch.tensor(7., dtype=DTYPE))
close("peso_preservado", w.detach(), torch.tensor(2., dtype=DTYPE))
print("L, dL/dw, dL/db:", loss.item(), w.grad.item(), b.grad.item())


## 3. Dois caminhos até a mesma entrada
Se u=x² e L=u+3x em x=2, as contribuições 4 e 3 se somam: dL/dx=7.

In [ ]:
x = torch.tensor(2., dtype=DTYPE, requires_grad=True)
L = x.square() + 3*x
(g,) = torch.autograd.grad(L, x)
close("caminhos_somados", g, torch.tensor(7., dtype=DTYPE))
ok("grad_retorna_sem_preencher", x.grad is None)
print("derivada total:", g.item())


## 4. Saída vetorial e Jacobiano manual

Defina f(x)=(x₁²+x₂, x₁x₂, sin(x₂)). Em x=(2,0), J=[[4,1],[0,2],[0,1]]. Com v=(1,-2,3), Jᵀv=(4,0). O vetor v é fixo e tem o shape da saída.

In [ ]:
def f(x):
    return torch.stack((x[0].square()+x[1], x[0]*x[1], torch.sin(x[1])))
x0 = torch.tensor([2., 0.], dtype=DTYPE)
J_manual = torch.tensor([[4., 1.], [0., 2.], [0., 1.]], dtype=DTYPE)
v = torch.tensor([1., -2., 3.], dtype=DTYPE)
x = x0.clone().requires_grad_()
y = f(x)
(gv,) = torch.autograd.grad(y, x, grad_outputs=v)
close("forward_vetorial", y.detach(), torch.tensor([4., 0., 0.], dtype=DTYPE))
close("vjp_manual", gv, J_manual.T @ v)
close("vjp_valor", gv, torch.tensor([4., 0.], dtype=DTYPE))
ok("shape_vjp_entrada", gv.shape == x.shape and v.shape == y.shape)
print("J^T v =", gv.tolist())


## 5. Contraprovas: sem vetor e vetor incompatível
Um tensor com mais de um elemento não recebe semente implícita. A exceção de elemento único também é exercitada para distinguir numel de ndim.

In [ ]:
def expect_runtime(name, operation, fragment):
    try:
        operation()
    except RuntimeError as exc:
        ok(name, fragment.lower() in str(exc).lower())
    else:
        raise AssertionError(name + ": deveria falhar")
expect_runtime("vetor_sem_semente", lambda: f(x0.clone().requires_grad_()).backward(), "scalar")
x = x0.clone().requires_grad_()
expect_runtime("semente_shape_errado", lambda: torch.autograd.grad(f(x), x, grad_outputs=torch.ones(2, dtype=DTYPE)), "shape")
single = torch.tensor([3.], dtype=DTYPE, requires_grad=True)
single.square().backward()
close("um_elemento_nao_zero_dim", single.grad, torch.tensor([6.], dtype=DTYPE))
print("Duas falhas esperadas confirmadas; backward de shape (1,) aceito.")


## 6. Mesma VJP, duas APIs
Recriamos o forward em cada consulta. backward grava em .grad; autograd.grad retorna uma tupla. v equivale à derivada de uma soma ponderada com coeficientes constantes.

In [ ]:
xb = x0.clone().requires_grad_()
f(xb).backward(v)
xs = x0.clone().requires_grad_()
(gs,) = torch.autograd.grad((v*f(xs)).sum(), xs)
close("backward_mesma_vjp", xb.grad, gv)
close("soma_ponderada_mesma_vjp", gs, gv)
ok("grad_nao_acumulou", xs.grad is None)
print("backward e soma ponderada:", xb.grad.tolist(), gs.tolist())


## 7. Uma direção na saída não é uma direção na entrada
VJP Jᵀv tem 2 elementos. JVP Ju, para u=(0,1), tem 3. Aqui calculamos Ju pela matriz manual para expor a diferença.

In [ ]:
u = torch.tensor([0., 1.], dtype=DTYPE)
jvp_manual = J_manual @ u
close("jvp_manual_valor", jvp_manual, torch.tensor([1., 2., 1.], dtype=DTYPE))
close("dualidade", torch.dot(v, jvp_manual), torch.dot(gv, u))
rows = []
for e in torch.eye(3, dtype=DTYPE):
    xi = x0.clone().requires_grad_()
    rows.append(torch.autograd.grad(f(xi), xi, grad_outputs=e)[0])
J_rebuilt = torch.stack(rows)
close("jacobiano_reconstruido", J_rebuilt, J_manual)
print("Jacobiano reconstruído:", J_rebuilt.tolist())


## 8. Soma, média e ponderação
Para perdas por exemplo ℓᵢ=(θ-aᵢ)²/2, a=(1,2,4), θ=0, os gradientes individuais são (-1,-2,-4). Média usa v=(1/3,1/3,1/3).

In [ ]:
a = torch.tensor([1., 2., 4.], dtype=DTYPE)
def batch_grad(weights):
    theta = torch.tensor(0., dtype=DTYPE, requires_grad=True)
    losses = (theta-a).square()/2
    return torch.autograd.grad(losses, theta, grad_outputs=weights)[0]
g_sum = batch_grad(torch.ones_like(a))
g_mean = batch_grad(torch.ones_like(a)/a.numel())
g_weighted = batch_grad(torch.tensor([0.2, 0.3, 0.5], dtype=DTYPE))
close("soma_lote", g_sum, torch.tensor(-7., dtype=DTYPE))
close("media_lote", g_mean, torch.tensor(-7/3, dtype=DTYPE))
close("pesos_lote", g_weighted, torch.tensor(-2.8, dtype=DTYPE))
close("fator_media", g_sum, 3*g_mean)
print("soma, média, ponderada:", g_sum.item(), g_mean.item(), g_weighted.item())


## 9. O backward do broadcasting soma contribuições
Para Y=X+b, cada linha usa o mesmo b. Logo G_b=sum_i V_i, não a média. A redução desejada deve estar contida em V.

In [ ]:
X = torch.arange(6, dtype=DTYPE).reshape(3,2)
bias = torch.tensor([0.5, -0.5], dtype=DTYPE, requires_grad=True)
V = torch.tensor([[1.,2.],[3.,4.],[5.,6.]], dtype=DTYPE)
(db,) = torch.autograd.grad(X+bias, bias, grad_outputs=V)
close("bias_soma_linhas", db, torch.tensor([9., 12.], dtype=DTYPE))
ok("bias_media_errada_detectada", not torch.allclose(db, V.mean(dim=0)))
print("VJP do bias:", db.tolist())


## 10. MLP 3 → 4 → 2: gradientes manuais em NumPy

A=tanh(XW₁+b₁), S=AW₂+b₂ e L=mean((S-T)²)/2. B=5, C=2, portanto G_S=(S-T)/(BC). Dados e pesos são gerados uma única vez em NumPy e copiados. Não há treinamento nem seleção de modelo.

In [ ]:
B, D, H, C = 5, 3, 4, 2
Xn = rng.normal(size=(B,D))
Tn = rng.normal(size=(B,C))
Pn = {"W1": rng.normal(scale=0.3,size=(D,H)), "b1": np.zeros(H),
      "W2": rng.normal(scale=0.3,size=(H,C)), "b2": np.zeros(C)}
An = np.tanh(Xn @ Pn["W1"] + Pn["b1"])
Sn = An @ Pn["W2"] + Pn["b2"]
Ln = np.mean((Sn-Tn)**2)/2
GS = (Sn-Tn)/(B*C)
GZ = (GS @ Pn["W2"].T)*(1-An**2)
manual = {"W1": Xn.T @ GZ, "b1": GZ.sum(axis=0),
          "W2": An.T @ GS, "b2": GS.sum(axis=0)}
Pt = {k: torch.tensor(val,dtype=DTYPE,requires_grad=True) for k,val in Pn.items()}
Xt, Tt = torch.tensor(Xn,dtype=DTYPE), torch.tensor(Tn,dtype=DTYPE)
def mlp_loss(params):
    A = torch.tanh(Xt @ params["W1"] + params["b1"])
    S = A @ params["W2"] + params["b2"]
    return (S-Tt).square().mean()/2
Lt = mlp_loss(Pt)
auto = dict(zip(Pt, torch.autograd.grad(Lt, tuple(Pt.values()))))
close("loss_numpy", Lt.detach(), torch.tensor(Ln,dtype=DTYPE))
errors = {}
for name in Pt:
    close("mlp_"+name, auto[name], torch.tensor(manual[name],dtype=DTYPE))
    errors[name] = float(np.max(np.abs(auto[name].numpy()-manual[name])))
ok("mlp_grad_retorno", all(p.grad is None for p in Pt.values()))
print("L =", Lt.item(), "erros por parâmetro =", errors)


## 11. Contraprova: esquecer a média sobre as saídas
Dividir por B quando a loss usa BC duplica todos os gradientes nesta fixture C=2. Autograd segue a loss que foi programada.

In [ ]:
wrong_GS = (Sn-Tn)/B
wrong_W2 = An.T @ wrong_GS
close("defeito_fator_C", torch.tensor(wrong_W2,dtype=DTYPE), C*auto["W2"])
ok("defeito_reducao_detectado", not np.allclose(wrong_W2, manual["W2"]))
print("fator incorreto:", C)


## 12. Uma diferença central direcional como controle independente
Normalizamos uma direção conjunta de todos os parâmetros. A aproximação [L(θ+hu)-L(θ-hu)]/(2h) deve conferir com a soma dos produtos G·u. Isto não substitui a auditoria sistemática da Aula 05.

In [ ]:
direction = {k: torch.tensor(rng.normal(size=p.shape),dtype=DTYPE) for k,p in Pt.items()}
norm = torch.sqrt(sum((v*v).sum() for v in direction.values()))
direction = {k:v/norm for k,v in direction.items()}
h = 1e-5
base = {k:torch.tensor(v,dtype=DTYPE) for k,v in Pn.items()}
plus = {k:base[k]+h*direction[k] for k in base}
minus = {k:base[k]-h*direction[k] for k in base}
finite_difference = (mlp_loss(plus)-mlp_loss(minus))/(2*h)
analytic_direction = sum((auto[k]*direction[k]).sum() for k in auto)
directional_error = (finite_difference-analytic_direction).abs().item()
close("direcional_independente", finite_difference, analytic_direction, atol=1e-9, rtol=1e-8)
print("erro direcional:", directional_error)


## 13. Semente variável: um detalhe matemático decisivo
Para y=x² e v=x em x=2, a VJP vale v·2x=8. Mas d(v(x)y(x))/dx=d(x³)/dx=12. grad_outputs aplica o cotangente recebido; não adiciona a derivada dos coeficientes como faria a função produto.

In [ ]:
q = torch.tensor(2.,dtype=DTYPE,requires_grad=True)
(local_vjp,) = torch.autograd.grad(q.square(), q, grad_outputs=q)
r = torch.tensor(2.,dtype=DTYPE,requires_grad=True)
(product_grad,) = torch.autograd.grad(r*r.square(), r)
close("cotangente_local", local_vjp, torch.tensor(8.,dtype=DTYPE))
close("produto_coeficiente_variavel", product_grad, torch.tensor(12.,dtype=DTYPE))
print("VJP local e derivada do produto:", local_vjp.item(), product_grad.item())


## 14. Auditoria e limites
Este notebook contém uma contraprova de saída vetorial sem semente e outra de shape incompatível. Não há exceções ignoradas genericamente, nem otimização dos pesos.

In [ ]:
ok("todos_float64_cpu", all(p.dtype==DTYPE and p.device.type=="cpu" for p in Pt.values()))
ok("entradas_nao_treinadas", not Xt.requires_grad and not Tt.requires_grad)
print(f"{len(checks)}/{len(checks)} verificações aprovadas")
print("Maior erro NumPy/autograd:", max(errors.values()))
print("Erro direcional:", directional_error)


## Exercícios para explorar

1. Troque v por (0,1,0). Antes de executar, preveja (0,2), a segunda linha do Jacobiano.
2. Troque mean por sum na MLP: todos os gradientes serão multiplicados por B·C=10.
3. Substitua o cotangente do bias por V/3: o resultado passa de (9,12) para (3,4).

Respostas e derivações completas estão na aula. A próxima aula trata o ciclo de vida do grafo, acúmulo em .grad e desconexões. As células desta aula recriam os forwards quando fazem novas consultas.

Referências técnicas: [Autograd mechanics 2.6](https://docs.pytorch.org/docs/2.6/notes/autograd.html), [autograd.grad 2.14](https://docs.pytorch.org/docs/2.14/generated/torch.autograd.grad.html), [backward 2.14](https://docs.pytorch.org/docs/2.14/generated/torch.Tensor.backward.html). Consultadas em 9 de setembro de 2026; somente a API comum exercitada em 2.6 é usada aqui.